# SPIN Healthcare — Milestone 1: coupling-collapse check (SPIN Step 1)

**Contribution 3 gate.** Does a non-trivial healthcare fairness∩privacy coupled set exist, or does it
collapse toward chance in a narrow clinical domain? This runs **SPIN Step 1 only** (importance → coupled
indices) on the three healthcare datasets, and emits `coupling_stats_health.json`. If the intersection is
well above chance and of comparable order to the general-domain baseline → proceed to Milestone 2 (Steps
2–4). If it collapses → that is a finding; pivot.

**Only the three datasets change** vs the faithful `author-sample` Step 1. Everything else is verbatim:
importance metric `|W|·Σ|∇L|`, authors' `get_set_difference_mask`, `p=q=5e-7`, MLP target, grouped-layer
scoring, fp16, `MAXLEN=1024`, per-group checkpointing.

- **D_f (fairness)** = EquityMedQA, 7 subsets, strict-unbiased `(question, answer)` targets (ALL raters
  "No bias"), 128 **stratified** across subsets.
- **D_p (privacy / patient rights)** = MedSafetyBench category_4 demonstrations
  `(harmful_medical_request, safe_response)`, 128.
- **D_g (general medical)** = PubMedQA `pqa_labeled` `(question, long_answer)`, 128.

**Reference points for the verdict:** general-domain 7B baseline `|N_f∩N_p|=1877`, clean `889`,
code-positional `1201`; and chance overlap `≈ Σ |N_f|·|N_p| / (MLP weights)`.

**Inputs required on Kaggle:** upload `equitymedqa_ratings.xlsx` as a Dataset input. Internet ON
(clones MedSafetyBench, pulls PubMedQA + Qwen2-7B). GPU **T4×2**. Runtime ~20–25 min.

**Output:** `/kaggle/working/coupling_stats_health.json` (+ `cal_{fair,priv,general}_health.csv`).

## Step 0 — env + config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys
# authors' pins: transformers 4.46.3 + datasets 2.18.0 (bit-reproducible shuffle). openpyxl for the xlsx.
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0',
                'sentencepiece','openpyxl'], check=False)
import random, gc, time, json, glob
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

#MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODEL_ID='Qwen/Qwen2-7B-Instruct'
NSAMPLES=128
TARGET='mlp'
MAXLEN=1024          # safety cap (authors don't truncate; calibration samples are short)
LAYER_GROUP=4
P=5e-7; Q=5e-7       # authors' Qwen2-7B extraction ratio (identical to faithful Step 1)
SEED=0
DTYPE=torch.float16
# 7 EquityMedQA subsets to KEEP (matched on the parenthetical tag in `dataset`)
SUBSET_ORDER=['OMAQ','EHAI','FBRT-Manual','FBRT-LLM','TRINDS','CC-Manual','CC-LLM']
# general-domain 7B author-sample baseline (reference point for the gate)
GEN_BASE = {
    'Qwen/Qwen2-7B-Instruct':      {'n_fp': 1877, 'n_fp_notg': 889, 'code': 1201},
    'Qwen/Qwen2.5-0.5B-Instruct':  {'n_fp': 101,  'n_fp_notg': 56,  'code': 86},
}.get(MODEL_ID)  # None -> defer verdict, report vs-chance only
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
print('config ready | model', MODEL_ID, '| p=q', P, '| subsets', SUBSET_ORDER)

## Step 1 — build the three healthcare calibration sets

Each is cast into the reproduction's `(prompt, response)` CSV schema so the authors' loader semantics
(prompt ⊕ response[1:], prompt masked to -100, loss on response) are reused unchanged.

### 1a — D_f: EquityMedQA strict-unbiased, stratified across 7 subsets

In [ ]:
# locate the uploaded xlsx (upload equitymedqa_ratings.xlsx as a Kaggle Dataset)
EQUITY_XLSX=None
for cand in glob.glob('/kaggle/input/**/equitymedqa_ratings.xlsx', recursive=True)+['equitymedqa_ratings.xlsx']:
    if os.path.exists(cand): EQUITY_XLSX=cand; break
assert EQUITY_XLSX, 'Upload equitymedqa_ratings.xlsx as a Kaggle Dataset input and re-run.'
print('xlsx:', EQUITY_XLSX)

raw=pd.read_excel(EQUITY_XLSX, sheet_name='Independent ratings')
def subset_of(name):
    for tag in SUBSET_ORDER:
        if f'({tag})' in str(name): return tag   # '(OMAQ)' avoids catching 'Mixed MMQA-OMAQ'
    return None
raw=raw.dropna(subset=['question_text','answer_text','bias_presence']).copy()
raw['subset']=raw['dataset'].apply(subset_of)
df=raw[raw['subset'].notna()].copy()
print('rated rows in the 7 subsets:', len(df))

# strict-unbiased pair = (question,answer) where EVERY rater row says 'No bias'
grp=df.groupby(['subset','question_text','answer_text'])['bias_presence']
pair=grp.agg(lambda s:(s=='No bias').all()).reset_index(name='strict')
pair['n_raters']=grp.size().values
strict=pair[pair['strict']].copy()
avail={s:int((strict['subset']==s).sum()) for s in SUBSET_ORDER}
print('strict-unbiased pairs available per subset:', avail, '| total', sum(avail.values()))

# proportional (largest-remainder) allocation of 128 across subsets, capped by availability
total_avail=sum(avail.values()); N_CAL=min(NSAMPLES,total_avail)
raw_alloc={s:(N_CAL*avail[s]/total_avail if total_avail else 0) for s in SUBSET_ORDER}
alloc={s:int(np.floor(raw_alloc[s])) for s in SUBSET_ORDER}
rem=N_CAL-sum(alloc.values())
order=sorted(SUBSET_ORDER,key=lambda s:raw_alloc[s]-alloc[s],reverse=True); i=0
while rem>0 and i<10000:
    s=order[i%len(order)]
    if alloc[s]<avail[s]: alloc[s]+=1; rem-=1
    i+=1
print('allocation (target 128):', alloc, '| sum', sum(alloc.values()))

parts=[]
for s in SUBSET_ORDER:
    sub=strict[strict['subset']==s]
    k=min(alloc[s],len(sub))
    if k>0: parts.append(sub.sample(n=k,random_state=SEED))
cal_f=pd.concat(parts).rename(columns={'question_text':'prompt','answer_text':'response'})[['prompt','response','subset']]
cal_f.to_csv('cal_fair_health.csv',index=False)
print('D_f calibration ->', len(cal_f), 'pairs written to cal_fair_health.csv')

### 1b — D_p: MedSafetyBench category_4 demonstrations

In [ ]:
if not os.path.exists('med-safety-bench'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI4LIFE-GROUP/med-safety-bench.git'], check=True)
files=[]
for split in ['train','test']:
    for src in ['gpt4','llama2']:
        p=f'med-safety-bench/datasets/{split}/{src}/med_safety_demonstrations_category_4.csv'
        if os.path.exists(p): files.append(p)
assert files, 'category_4 demonstration CSVs not found after clone.'
print('demo files:', files)
dp=[]
for p in files:
    d=pd.read_csv(p)
    cols={c.lower().strip():c for c in d.columns}
    qc=cols.get('harmful_medical_request'); ac=cols.get('safe_response')
    if qc is None or ac is None:   # fallback: leading unnamed index col misparse
        d=pd.read_csv(p,index_col=0); cols={c.lower().strip():c for c in d.columns}
        qc=cols.get('harmful_medical_request'); ac=cols.get('safe_response')
    dp.append(d[[qc,ac]].rename(columns={qc:'prompt',ac:'response'}))
dp=pd.concat(dp).dropna().drop_duplicates(['prompt','response']).reset_index(drop=True)
print('category_4 (harmful,safe) pairs available:', len(dp))
cal_p=dp.sample(n=min(NSAMPLES,len(dp)),random_state=SEED)[['prompt','response']]
cal_p.to_csv('cal_priv_health.csv',index=False)
print('D_p calibration ->', len(cal_p), 'pairs written to cal_priv_health.csv')

### 1c — D_g: PubMedQA (pqa_labeled) general medical capability

In [ ]:
try:
    pq=load_dataset('pubmed_qa','pqa_labeled',split='train')
except Exception as e:
    print('retry with trust_remote_code:', e)
    pq=load_dataset('pubmed_qa','pqa_labeled',split='train',trust_remote_code=True)
pqd=pq.shuffle(seed=SEED).select(range(min(NSAMPLES,len(pq))))
cal_g=pd.DataFrame({'prompt':[str(x) for x in pqd['question']],
                    'response':[str(x) for x in pqd['long_answer']]})
cal_g.to_csv('cal_general_health.csv',index=False)
print('D_g calibration ->', len(cal_g), 'pairs written to cal_general_health.csv')

## Step 2 — model + calibration batches (authors' loader semantics)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'})
model.eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model split loaded | entry', GEN_DEV); memline('loaded')

def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]

# authors' loader semantics (utils.get_loaders disentangle=True), applied to our fixed calibration CSVs:
#   inp = prompt + response[:,1:] (unconditional BOS drop); tar = inp with prompt tokens masked to -100.
def make_batches_from_csv(path):
    d=pd.read_csv(path); out=[]
    for i in range(len(d)):
        ep=tokenizer(str(d['prompt'].iloc[i]),   return_tensors='pt').input_ids
        er=tokenizer(str(d['response'].iloc[i]), return_tensors='pt').input_ids[:,1:]
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out
BATCHES={'f':make_batches_from_csv('cal_fair_health.csv'),
         'p':make_batches_from_csv('cal_priv_health.csv'),
         'g':make_batches_from_csv('cal_general_health.csv')}
print('batches | f',len(BATCHES['f']),'| p',len(BATCHES['p']),'| g',len(BATCHES['g']))

## Step 3 — score group-by-group, coupled indices + set sizes (authors' exact selection)

`get_set_difference_mask`, `coupled_indices_for_matrix` are verbatim from the reproduction. `coupling_set_sizes`
adds true-set accounting (|N_f|,|N_p|,|N_g|,|N_f∩N_p|,clean) and per-matrix numel for the chance baseline.

In [ ]:
names=target_names(model); nL=model.config.num_hidden_layers
GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]
CK='/kaggle/working/health_coupled_ckpt.pt'

# ---- authors' EXACT SPIN.py logic (verbatim) ----
def get_set_difference_mask(p, q, W_metric1, W_metric2):
    top_p=int(p*W_metric1.shape[1]*W_metric1.shape[0])
    top_q=int(q*W_metric2.shape[1]*W_metric2.shape[0])
    top_p_indices=torch.topk(W_metric1.flatten(), top_p, largest=True)[1]
    top_q_indices=torch.topk(W_metric2.flatten(), top_q, largest=True)[1]
    unique_p=torch.unique(top_p_indices); unique_q=torch.unique(top_q_indices)
    mask_only_metric1=~torch.isin(unique_p, unique_q)
    mask_only_metric2=~torch.isin(unique_q, unique_p)
    mask_intersection=~(torch.ones_like(unique_p).bool() & mask_only_metric1 & mask_only_metric2)
    return mask_only_metric1, mask_only_metric2, mask_intersection, unique_q

def coupled_indices_for_matrix(W_general, W_fair, W_priv):
    _, m1, _, u1 = get_set_difference_mask(P, Q, W_general, W_fair)
    _, m2, _, u2 = get_set_difference_mask(P, Q, W_general, W_priv)
    return u1[m1 & m2]

def coupling_set_sizes(W_general, W_fair, W_priv):
    # true SET accounting (independent of the authors' positional mask) + numel for chance baseline
    def tks(W, ratio):
        k=int(ratio*W.shape[0]*W.shape[1])
        return set(torch.unique(torch.topk(W.flatten(), k, largest=True)[1]).tolist()) if k>=1 else set()
    Sg=tks(W_general,P); Sf=tks(W_fair,Q); Sp=tks(W_priv,Q); fp=Sf & Sp
    return {'ng':len(Sg),'nf':len(Sf),'np':len(Sp),'n_fp':len(fp),'n_fp_notg':len(fp-Sg),
            'numel':int(W_general.shape[0]*W_general.shape[1])}

@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}
    del acc; gc.collect(); torch.cuda.empty_cache()
    return imp

if os.path.exists(CK):
    st=torch.load(CK); coupled=st['coupled']; cstats=st.get('cstats',{}); start=st['next_group']
    print('resuming from group', start)
else:
    coupled={}; cstats={}; start=0

model.gradient_checkpointing_enable(); model.config.use_cache=False
for gi in range(start,len(GROUPS)):
    grp=GROUPS[gi]; gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
    t0=time.time()
    imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
    for n in gnames:
        fi=coupled_indices_for_matrix(imp_g[n], imp_f[n], imp_p[n])
        ss=coupling_set_sizes(imp_g[n], imp_f[n], imp_p[n]); ss['code']=int(len(fi)); cstats[n]=ss
        if len(fi): coupled[n]=[int(x) for x in fi.tolist()]
    del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
    torch.save({'coupled':coupled,'cstats':cstats,'next_group':gi+1}, CK)
    print(f'  group {gi+1}/{len(GROUPS)} done ({time.time()-t0:.0f}s) | code-coupled so far {sum(len(v) for v in coupled.values())}'); memline()
model.gradient_checkpointing_disable()
for p in model.parameters(): p.requires_grad_(False)
print('scoring complete.')

## Step 4 — aggregate, reference points, verdict → `coupling_stats_health.json`

In [ ]:
import re as _re
def _layer(n):
    m=_re.search(r'layers\.(\d+)\.', n); return int(m.group(1)) if m else -1
def _proj(n):
    m=_re.search(r'mlp\.(\w+?)_proj', n); return m.group(1) if m else '?'

agg={k:int(sum(cs.get(k,0) for cs in cstats.values())) for k in ['ng','nf','np','n_fp','n_fp_notg','code']}
mlp_params=int(sum(model.get_parameter(n).numel() for n in names))
# chance overlap: per matrix E[|Sf∩Sp|] = |Sf|*|Sp|/numel, summed over matrices
chance=float(sum(cs['nf']*cs['np']/cs['numel'] for cs in cstats.values() if cs.get('numel')))

per_layer={}; per_proj={}
for n,idxs in coupled.items():
    per_layer[_layer(n)]=per_layer.get(_layer(n),0)+len(idxs)
    per_proj[_proj(n)]=per_proj.get(_proj(n),0)+len(idxs)

observed=agg['n_fp']
ratio_vs_chance=(observed/chance) if chance>0 else float('inf')
ratio_vs_general=observed/GEN_BASE['n_fp'] if GEN_BASE['n_fp'] else 0.0
# advisory verdict (Amina decides): well-posed if far above chance AND same order as general-domain
well_posed = (ratio_vs_chance>=100) and (ratio_vs_general>=0.2)
verdict = 'WELL-POSED — proceed to Milestone 2' if well_posed else 'REVIEW — possible collapse; inspect before proceeding'

out={'model':MODEL_ID,'p':P,'q':Q,'nsamples':NSAMPLES,'target':TARGET,
     'calibration':{'fair':len(BATCHES['f']),'priv':len(BATCHES['p']),'general':len(BATCHES['g'])},
     'aggregate':agg,                       # summed over MLP matrices: ng,nf,np,n_fp,n_fp_notg,code
     'coupled_total_code':int(sum(len(v) for v in coupled.values())),
     'mlp_params':mlp_params,
     'intervention_sparsity':(agg['code']/mlp_params if mlp_params else 0.0),
     'chance_overlap':chance,
     'reference_general_baseline':GEN_BASE,
     'ratio_vs_chance':ratio_vs_chance,'ratio_vs_general':ratio_vs_general,
     'verdict':verdict,
     'coupled_per_layer':{str(k):per_layer[k] for k in sorted(per_layer)},
     'coupled_per_proj':per_proj,
     'per_matrix':cstats}
json.dump(out, open('/kaggle/working/coupling_stats_health.json','w'), indent=1)

print('=== Milestone 1 — coupling-collapse check ===')
print(f"  |N_f|={agg['nf']}  |N_p|={agg['np']}  |N_g|={agg['ng']}   (summed over MLP matrices)")
print(f"  |N_f n N_p| = {agg['n_fp']}   clean (N_f n N_p)\\N_g = {agg['n_fp_notg']}   code-positional = {agg['code']}")
print(f"  chance overlap = {chance:.2f}   ->  observed / chance = {ratio_vs_chance:.1f}x")
print(f"  general-domain baseline |N_f n N_p| = {GEN_BASE['n_fp']} (clean {GEN_BASE['n_fp_notg']}, code {GEN_BASE['code']})")
print(f"  healthcare / general = {ratio_vs_general:.2f}")
print(f"  intervention sparsity = {out['intervention_sparsity']:.2e}  ({agg['code']} / {mlp_params})")
print(f"  coupled per projection: {per_proj}")
print(f"  VERDICT: {verdict}")
print('  -> coupling_stats_health.json')